In [1]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
from torch import nn
from sklearn.preprocessing import LabelEncoder


In [2]:
data = pd.read_csv("2-filled_data.csv")
data.describe()
          

,age at visit,ANIMALS,asscmem,srtfree,srttotal,tma,tmb,INDEPEND,RESIDENC,MARISTAT,...,DEP,OTHPSY,DOWNS,PARK,HYCEPH,BRNINJ,NEOP,COGOTH,COGOTH2,COGOTH3
count,8777.000000,8777.000000,8777.000000,8777.000000,8777.000000,8777.000000,8777.000000,8777.000000,8777.000000,8777.000000,...,8777.000000,8777.000000,8777.0,8777.000000,8777.000000,8777.000000,8777.000000,8777.000000,8777.000000,8777.000000
mean,73.793551,19.641677,13.713683,29.295545,47.243022,36.002051,91.742851,1.150507,1.097869,1.547681,...,0.135240,0.005127,0.0,0.010368,0.000456,0.009343,0.000798,0.023812,0.002165,0.000342
std,8.684403,5.782562,3.806465,7.125961,3.229885,19.127606,37.298829,0.496820,0.438690,1.064911,...,0.341999,0.071424,0.0,0.101300,0.021344,0.096210,0.028231,0.152472,0.046479,0.018486
min,-48.000000,1.000000,0.000000,0.000000,3.000000,10.000000,19.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,69.000000,16.000000,12.000000,27.000000,48.000000,27.000000,65.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,74.000000,20.000000,14.000000,30.000000,48.000000,32.000000,92.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,79.000000,23.000000,16.000000,33.000000,48.000000,38.000000,99.000000,1.000000,1.000000,2.000000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,100.000000,47.000000,21.000000,48.000000,48.000000,180.000000,300.000000,9.000000,9.000000,9.000000,...,1.000000,1.000000,0.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [3]:
df=data.copy()

In [4]:
cat_cols = df.select_dtypes(include="object").columns
cat_cols

Index(['OASISID', 'meds_1', 'meds_2', 'meds_3', 'meds_4', 'dx1', 'dx2', 'dx3',
       'dx4', 'dx5', 'all_meds'],
      dtype='object')

In [5]:
df.drop(["all_meds","OASISID"],axis=1,inplace=True)


In [6]:
cat_cols = df.select_dtypes(include="object").columns
cat_cols

Index(['meds_1', 'meds_2', 'meds_3', 'meds_4', 'dx1', 'dx2', 'dx3', 'dx4',
       'dx5'],
      dtype='object')

In [7]:
def label_encoder(data):
    encoder = LabelEncoder()
    data = encoder.fit_transform(data)
    return data

In [8]:
def embedding(dimension,data_size,data):
    embedding = nn.Embedding(num_embeddings=data_size, embedding_dim=dimension)
    encoded_tensor = torch.tensor(data, dtype=torch.long)
    embedded_data = embedding(encoded_tensor)
    return embedded_data

In [9]:
# Tüm ilaçları alt alta yazdırdık
all_meds = pd.concat([df[col] for col in cat_cols[:4]],axis=0)
print(len(all_meds))
# Nan değerleri sildik 
all_meds.dropna(inplace=True)
print(len(all_meds))
meds_uniq = all_meds.unique()
print(len(meds_uniq)) # uniq ilaç sayısı
print(meds_uniq) # uniq ilaçlar

meds_encode = label_encoder(meds_uniq)
meds_len = len(meds_encode)
meds_encode

35108
27592
2672
['lisinopr' 'aspirin' 'lisinopril' ... 'epinastine hcl ophthalmic 0.05%'
 'telmisartan/hctz- (micardis hct)' 'altrum ultra probiotics']


array([1453,  210, 1454, ...,  873, 2292,  122], shape=(2672,))

In [10]:
embedded_data = embedding(16,meds_len,meds_encode)
embedded_data.shape  

torch.Size([2672, 16])

In [11]:
embedded_np = embedded_data.detach().numpy()
embedded_meds_df = pd.DataFrame(embedded_np)

embedded_meds_df.insert(0, "medicine", meds_uniq)
embedded_meds_df.head(10)

,medicine,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,lisinopr,0.486680,0.028075,0.536239,0.628803,-1.278358,-0.607274,1.654251,-1.267180,0.882840,-0.785149,0.448470,0.586418,0.246107,0.809650,0.350126,-1.694267
1,aspirin,0.618790,-0.344045,0.297618,-0.544550,-1.466511,-1.447107,-0.997070,0.062203,1.789916,-0.531344,0.793767,-1.178684,1.056968,0.190247,0.218834,0.171719
2,lisinopril,-0.221657,-0.415213,-0.649467,0.098007,-0.269497,0.322408,0.688707,-0.582546,0.392349,0.838462,-0.531084,0.218243,0.652024,0.622434,-0.655480,-0.098588
3,aspirin d,-0.443065,-0.907376,-0.972251,0.705977,-0.437357,2.178245,1.148185,-0.203080,1.473358,-0.023604,0.522352,0.380466,-1.128309,-0.099939,0.827298,0.284309
4,dhea,0.692102,-0.075731,-0.048704,1.088686,-1.792119,-0.408348,-0.003814,-1.006279,2.422332,1.485324,1.550920,-0.418506,-1.203722,-1.424733,0.374856,1.416118
5,ascorbic acid,-0.735257,-0.673179,-1.259889,-0.961147,0.328207,-0.578057,0.758425,-0.201051,0.938882,0.341054,-0.009462,0.828291,0.308684,-0.359790,-1.707558,0.432101
6,metronidazole,-1.120320,0.271440,2.212194,0.145013,0.654898,-1.441041,-0.715554,0.351940,-1.074552,-0.582057,0.922074,0.559621,-0.736861,-0.860959,0.252550,1.193391
7,ascorbic acid (c complex - vitamin c) d,-0.400726,0.834116,-0.758041,-0.727390,-1.404203,1.190181,-0.414975,-1.746920,1.397989,2.538436,0.684009,-0.554480,-0.291082,-0.823365,0.578051,-1.081927
8,combiven,0.656831,0.580377,-1.741677,0.395089,0.262778,-0.015143,1.745746,-1.276470,-1.050719,-0.350241,1.067720,-0.415538,0.934592,-0.417698,-0.196452,0.341368
9,hydrochlorothiazide,-0.591025,-0.105824,-0.109822,1.333440,0.318993,0.041442,0.136471,0.595176,0.625174,0.211719,-0.879791,-0.800442,-1.231438,-1.492603,2.540862,-0.635328


In [12]:
all_dx = pd.concat([df[col] for col in cat_cols[4:]],axis=0)
len(all_dx)


43885

In [13]:
all_dx.value_counts()

a                                    15927
.                                    13631
cognitively normal                    6263
ad dementia                           1124
active mood disorder                   951
                                     ...  
dat visuospatial, after                  1
remote pd drug-induced                   1
oth potentially transmissible dis        1
active dlbd                              1
dat w/oth (list b) contribut             1
Name: count, Length: 99, dtype: int64

In [14]:
all_dx = all_dx.replace("a",np.nan)
all_dx = all_dx.replace(".",np.nan)
all_dx.value_counts()

cognitively normal                   6263
ad dementia                          1124
active mood disorder                  951
remote mood disorder                  793
uncertain dementia                    464
                                     ... 
dat visuospatial, after                 1
remote pd drug-induced                  1
oth potentially transmissible dis       1
active dlbd                             1
dat w/oth (list b) contribut            1
Name: count, Length: 97, dtype: int64

In [15]:
all_dx.dropna(inplace=True)
len(all_dx)

12672

In [16]:
dx_uniq = all_dx.unique()
print(len(dx_uniq))
print(dx_uniq[:5]) 

dx_encode = label_encoder(dx_uniq)
dx_len = len(dx_encode)
dx_encode

97
['cognitively normal' 'uncertain dementia' 'ad dementia'
 'unc: ques. impairment' 'ad dem w/depresss, not contribut']


array([43, 93, 41, 92, 31, 21, 28, 65, 27, 26, 22, 64, 44, 60, 95, 39,  0,
       94, 63, 20, 19, 38, 33, 58, 62, 57, 30, 91, 56, 34, 29, 23, 24, 48,
       67, 59, 36, 73, 18, 96, 35, 53, 47, 46, 52, 25, 74, 37, 32, 40, 50,
       51, 61, 78, 12, 13,  2, 87, 86,  4, 83,  9, 79, 15, 80, 84, 81, 85,
        5, 90, 42, 10, 89, 11,  3, 14, 76, 17,  1, 45, 55, 71, 75,  7, 77,
       68, 16, 70, 66, 49,  8, 72, 88, 82, 69,  6, 54])

In [17]:
embedded_data = embedding(4,dx_len,dx_encode)
embedded_data.shape  

torch.Size([97, 4])

In [18]:
embedded_np = embedded_data.detach().numpy()
embedded_dx_df = pd.DataFrame(embedded_np)

embedded_dx_df.insert(0, "dx", dx_uniq)
embedded_dx_df.head(10)

,dx,0,1,2,3
0,cognitively normal,1.828455,-1.067179,0.020507,0.817286
1,uncertain dementia,-0.541249,-0.295089,0.162520,-1.080201
2,ad dementia,0.379209,1.968740,0.208887,-0.084191
3,unc: ques. impairment,-1.029063,0.898215,2.142868,-0.633426
4,"ad dem w/depresss, not contribut",1.202419,0.381768,-0.863395,-0.712107
5,"ad dem distrubed social, with",-0.475934,0.846513,1.844324,-0.903117
6,ad dem w/cvd contribut,-1.749217,-1.713619,-1.951430,1.150468
7,"non ad dem, other primary",-1.540570,-2.281170,1.690464,-0.779987
8,"ad dem visuospatial, with",0.372155,-2.115952,1.321824,-0.808572
9,"ad dem visuospatial, prior",0.835226,0.106223,-2.026739,-0.037334


In [19]:
embedded_dx_df.to_csv("7-embedded_dx1.csv", index=False)
embedded_meds_df.to_csv("7-embedded_meds1.csv",index=False)

In [20]:
embeddings_lookup = pd.read_csv('7-embedded_meds1.csv')
main_data = pd.read_csv('2-filled_data.csv')
main_data.drop("all_meds",axis=1,inplace=True)
meds_cols = ['meds_1', 'meds_2', 'meds_3', 'meds_4']

final_data = main_data.copy()

for col in meds_cols:
    col_values = final_data[col].fillna('UNKNOWN') # eksik yerlere unknown yazıyoruz
    final_data = final_data.drop(columns=[col])   # ardından o kolonu siliyoruz 
    embeddings_matrix = [] # matrix alacak listeyi oluşturuyoruz 
    for med_name in col_values:
        embedding_row = embeddings_lookup[embeddings_lookup['medicine'] == str(med_name)] # ilaç ismine denk gelen satırı alıyoruz 
        # eger ilac'ın karşılığı varsa matrix karşılığını alıyoruz 
        if len(embedding_row) > 0:
            embedding_vector = embedding_row.iloc[0, 1:].values
        # eger yoksa yani UNKNOWN ise değer 0 lardan oluşan bir matrix yapıyoruz 
        else:
            embedding_vector = np.zeros(16)
        # sırayla kolondaki tüm satırlara 16 kolonluk matrixini alıp listeye ekliyoruz 
        embeddings_matrix.append(embedding_vector)
    # Sonra bu 16 kolonluk matrixi dataframe yapıyoruz 
    embedding_df = pd.DataFrame(embeddings_matrix, 
                               columns=[f'{col}_emb_{i}' for i in range(16)])
    # Ardından birleştirme işlemini yapıyoruz 
    final_data = pd.concat([final_data.reset_index(drop=True), 
                           embedding_df.reset_index(drop=True)], axis=1)

# final_data.to_csv('meds_replaced.csv', index=False)
print(f"Başlangıç: {main_data.shape} -> Final: {final_data.shape}")


Başlangıç: (8777, 96) -> Final: (8777, 156)


In [21]:
embeddings_lookup = pd.read_csv('7-embedded_dx1.csv')

dx_cols = ['dx1', 'dx2', 'dx3', 'dx4','dx5']

final_data1 = final_data.copy()

for col in dx_cols:
    col_values = final_data1[col].fillna('UNKNOWN') # eksik yerlere unknown yazıyoruz
    final_data1 = final_data1.drop(columns=[col])   # ardından o kolonu siliyoruz 
    embeddings_matrix = [] # matrix alacak listeyi oluşturuyoruz 
    for dx_name in col_values:
        embedding_row = embeddings_lookup[embeddings_lookup['dx'] == str(dx_name)] 
        
        if len(embedding_row) > 0:
            embedding_vector = embedding_row.iloc[0, 1:].values
        # eger yoksa yani UNKNOWN ise değer 0 lardan oluşan bir matrix yapıyoruz 
        else:
            embedding_vector = np.zeros(4)
        # sırayla kolondaki tüm satırlara 4 kolonluk matrixini alıp listeye ekliyoruz 
        embeddings_matrix.append(embedding_vector)
    # Sonra bu 4 kolonluk matrixi dataframe yapıyoruz 
    embedding_df = pd.DataFrame(embeddings_matrix, 
                               columns=[f'{col}_emb_{i}' for i in range(4)])
    # Ardından birleştirme işlemini yapıyoruz 
    final_data1 = pd.concat([final_data1.reset_index(drop=True), 
                           embedding_df.reset_index(drop=True)], axis=1)


print(f"Başlangıç: {final_data.shape} -> Final: {final_data1.shape}")

Başlangıç: (8777, 156) -> Final: (8777, 171)


In [22]:
final_data1.to_csv('7-end1.csv', index=False)